# Fund Portfolio Analyzer — Interactive Demo

This notebook lets you interactively explore your fund portfolio data.
Run all cells to see the analysis, or modify parameters to experiment.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.io as pio
pio.renderers.default = 'notebook'

# Load config
from src.config import load_config
from src.data.manager import get_portfolio_data

config = load_config('config.yaml')
print(f"Loaded: {config.name} with {len(config.funds)} funds")

In [ ]:
# Fetch data (uses cache if available)
nav_df, benchmark_data, warnings = get_portfolio_data(config)
print(f"Data: {nav_df.shape[0]} dates × {nav_df.shape[1]} funds")
print(f"Date range: {nav_df.index[0]} ~ {nav_df.index[-1]}")
nav_df.head()

In [ ]:
# Normalized NAV plot
from src.reporting.charts import plot_nav_series
fund_names = {f.code: f.name for f in config.funds}
plot_nav_series(nav_df, fund_names)

In [ ]:
# Returns & Risk
from src.analysis.returns import compute_returns, compute_portfolio_returns
from src.analysis.risk import risk_metrics_table

daily_returns = compute_returns(nav_df)
port_returns = compute_portfolio_returns(nav_df, config.weights)
risk_table = risk_metrics_table(daily_returns, config.analysis.risk_free_rate)
risk_table

In [ ]:
# Drawdown chart
from src.reporting.charts import plot_drawdown
plot_drawdown(daily_returns, fund_names)

In [ ]:
# Correlation heatmap
from src.analysis.correlation import correlation_matrix, effective_n
from src.reporting.charts import plot_correlation_heatmap

corr_df = correlation_matrix(daily_returns)
print(f"Effective N: {effective_n(corr_df)} of {len(corr_df)} funds")
plot_correlation_heatmap(corr_df, fund_names)

In [ ]:
# Hierarchical clustering
from src.analysis.correlation import hierarchical_clustering
from src.reporting.charts import plot_dendrogram

linkage, labels = hierarchical_clustering(daily_returns)
plot_dendrogram(linkage, labels, fund_names)

In [ ]:
# Efficient frontier
from src.analysis.optimization import (
    expected_returns, covariance_matrix, efficient_frontier,
    max_sharpe_portfolio, risk_parity_portfolio, min_variance_portfolio
)
from src.reporting.charts import plot_efficient_frontier

rf = config.analysis.risk_free_rate
exp_ret = expected_returns(daily_returns)
cov = covariance_matrix(daily_returns)

frontier = efficient_frontier(exp_ret, cov, rf)
ms_weights = max_sharpe_portfolio(exp_ret, cov, rf)
rp_weights = risk_parity_portfolio(cov)
mv_weights = min_variance_portfolio(cov)

plot_efficient_frontier(frontier, daily_returns, fund_names, ms_weights, rp_weights, mv_weights, exp_ret, cov)

In [ ]:
# Weight comparison
comparison = pd.DataFrame({
    'Current': [config.weights.get(c, 0) for c in daily_returns.columns],
    'Max Sharpe': ms_weights,
    'Risk Parity': rp_weights,
    'Min Variance': mv_weights,
}, index=daily_returns.columns)
comparison.index = [fund_names.get(c, c) for c in comparison.index]
comparison.plot.bar(figsize=(10, 5), title='Weight Comparison: Current vs Optimal')
comparison.round(3)

In [ ]:
# Risk contribution pie
from src.reporting.charts import plot_risk_contribution
plot_risk_contribution(ms_weights, cov, list(daily_returns.columns), fund_names)

In [ ]:
# Portfolio performance
from src.reporting.charts import plot_portfolio_performance

bm_key = list(benchmark_data.keys())[0] if benchmark_data else None
bm_df = benchmark_data[bm_key].set_index('date').sort_index() if bm_key else None
bm_returns = bm_df['value'].pct_change().dropna() if bm_df is not None else None

plot_portfolio_performance(port_returns, bm_returns, bm_key or 'Benchmark')